In [6]:
import sys
sys.path.append('..')

import os
import re
import glob
import pandas as pd

from nnspike.utils import extract_video_frames
from nnspike.data import create_label_dataframe, sort_by_frames_number, label_dataset_by_opencv, label_dataset_by_model, augment_dataset, set_spike_status
from nnspike.constants import  ROI_CNN

course = "right" # "right" or "left"

## Extract Frames from Videos

In [7]:
def get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250910/videos/", filter_timestamp=None):
    """
    Get all AVI files from the specified directory with their timestamps.
    
    Args:
        directory_path (str): Path to the directory containing AVI files
        filter_timestamp (str, optional): If set, only return files matching this timestamp pattern (supports wildcards with *)
    
    Returns:
        list: List of tuples containing (file_path, timestamp)
    """
    import os
    import glob
    import re
    import fnmatch

    # Use glob to find all .avi files in the directory
    avi_files = glob.glob(os.path.join(directory_path, "*.avi"))
    avi_files = [path.replace("\\", "/") for path in avi_files]
    
    # Sort the files for consistent ordering
    avi_files.sort()
    
    # Extract timestamps and create tuples
    result = []
    for avi_file in avi_files:
        # Extract filename without extension
        filename = os.path.basename(avi_file)
        filename_no_ext = os.path.splitext(filename)[0]
        
        # Extract timestamp from filename (assuming format: timestamp_picamera.avi)
        # This will extract the part before '_picamera'
        timestamp_match = re.match(r'^(\d{14})_.*', filename_no_ext)
        if timestamp_match:
            timestamp = timestamp_match.group(1)
        else:
            # If timestamp pattern not found, use the full filename without extension
            timestamp = filename_no_ext

        # If filter_timestamp is set, only include matching files using pattern matching
        if filter_timestamp is None or fnmatch.fnmatch(timestamp, filter_timestamp):
            result.append((avi_file, timestamp))
    
    return result

def extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250910/frames/"):
    """
    Extract frames from all AVI files and save them to folders named by timestamp.
    
    Args:
        avi_files_with_timestamps (list): List of tuples containing (file_path, timestamp)
        base_output_dir (str): Base directory where frame folders will be created
    
    Returns:
        list: List of tuples containing (output_directory, timestamp)
    """
    output_directories_with_timestamps = []
    
    for avi_file, timestamp in avi_files_with_timestamps:
        # Create output directory path
        output_dir = os.path.join(base_output_dir, timestamp)
        
        # Create directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Add trailing slash for extract_video_frames function
        output_dir_with_slash = output_dir + "/"
        
        filename = os.path.basename(avi_file)
        print(f"Extracting frames from {filename} to {output_dir_with_slash}")
        
        try:
            # Extract frames using the nnspike utility function
            extract_video_frames(avi_file, output_dir_with_slash)
            print(f"✓ Successfully extracted frames to {timestamp}/")
            # Add successful output directory and timestamp to the list
            output_directories_with_timestamps.append((output_dir, timestamp))
        except Exception as e:
            print(f"✗ Error extracting frames from {filename}: {str(e)}")
    
    return output_directories_with_timestamps



In [8]:
# Get all AVI files with their timestamps
avi_files_with_timestamps = get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250910/videos/")
avi_files_with_timestamps

[('C:/Users/MSAD/github/nnspike/storage/20250910/videos/20250910112636_picamera.avi',
  '20250910112636')]

In [9]:
    # Extract frames from all AVI files
if avi_files_with_timestamps:
    print("\nStarting frame extraction...")
    output_dirs_with_timestamps = extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250910/frames/")
    print("\nFrame extraction completed!")
    print(f"Successfully created {len(output_dirs_with_timestamps)} output directories:")
    for output_dir, timestamp in output_dirs_with_timestamps:
        print(f"  - {output_dir} (timestamp: {timestamp})")
else:
    print("No AVI files found to process.")
    output_dirs_with_timestamps = []
    
output_dirs_with_timestamps


Starting frame extraction...
Extracting frames from 20250910112636_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250910/frames/20250910112636/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250910\frames\20250910112636
✓ Successfully extracted frames to 20250910112636/

Frame extraction completed!
Successfully created 1 output directories:
  - C:/Users/MSAD/github/nnspike/storage/20250910/frames/20250910112636 (timestamp: 20250910112636)
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250910\frames\20250910112636
✓ Successfully extracted frames to 20250910112636/

Frame extraction completed!
Successfully created 1 output directories:
  - C:/Users/MSAD/github/nnspike/storage/20250910/frames/20250910112636 (timestamp: 20250910112636)


[('C:/Users/MSAD/github/nnspike/storage/20250910/frames/20250910112636',
  '20250910112636')]

In [10]:
import os
for output_dir, timestamp in output_dirs_with_timestamps:
    print(f"Output directory: {output_dir} (timestamp: {timestamp})")
    
    course = "right"
    label_df = create_label_dataframe(output_dir +"/*", course)
    if label_df is None or label_df.empty:
        print(f"Warning: label_df is empty for timestamp {timestamp}, skipping.")
        continue
    print("label_df type:", type(label_df))
    print("label_df head:", getattr(label_df, "head", lambda: "no head")())
    label_df = sort_by_frames_number(label_df)
    label_df = label_dataset_by_opencv(label_df, ROI_CNN, 80)
    
    sensor_path = f"C:/Users/MSAD/github/nnspike/storage/20250910/sensor_data/{timestamp}_sensor_log.csv"
    if not os.path.exists(sensor_path):
        print(f"Warning: Sensor log not found for timestamp {timestamp}, skipping.")
        continue
    
    status_df = pd.read_csv(sensor_path)
    df = set_spike_status(label_df, status_df)
    
    # Export to a csv file
    df.to_csv(f"C:/Users/MSAD/github/nnspike/storage/20250910/labels/{timestamp}_label.csv", index=False)

Output directory: C:/Users/MSAD/github/nnspike/storage/20250910/frames/20250910112636 (timestamp: 20250910112636)
label_df type: <class 'pandas.core.frame.DataFrame'>
label_df head:                                           image_path  mode  target_x  left_x  \
0  C:/Users/MSAD/github/nnspike/storage/20250910/...   NaN       NaN     NaN   
1  C:/Users/MSAD/github/nnspike/storage/20250910/...   NaN       NaN     NaN   
2  C:/Users/MSAD/github/nnspike/storage/20250910/...   NaN       NaN     NaN   
3  C:/Users/MSAD/github/nnspike/storage/20250910/...   NaN       NaN     NaN   
4  C:/Users/MSAD/github/nnspike/storage/20250910/...   NaN       NaN     NaN   

   right_x  motor_a_speed  motor_b_speed  motor_a_relative_position  \
0      NaN              0              0                          0   
1      NaN              0              0                          0   
2      NaN              0              0                          0   
3      NaN              0              0             

Processing: 100%|██████████| 376/376 [00:10<00:00, 36.95it/s]
